[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/02_tracking/B3_progress_indicators.ipynb)

# B3: Progress Indicators

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Calculate age metrics** for projects in review
2. **Identify stalled projects** using threshold rules
3. **Predict completion dates** based on historical patterns
4. **Create early warning systems** for delayed projects

## Why This Matters

Some projects sit in review for years while others move quickly. Understanding why helps:
- Identify systemic delays (staffing, policy issues)
- Flag projects that need attention
- Set realistic expectations for developers
- Measure improvement over time

## Stalled Project Definition

A project is "stalled" if:
- In review > 180 days without status change
- Corrections pending > 90 days
- Building permit issued > 730 days without CO

These thresholds can be calibrated based on local norms.

---

## Overview

Calculate progress indicators and identify stalled projects.

**Indicators:**
- Time between stages
- Stalled project flags (>180 days)
- Predicted completion dates

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 2. Load Project Data

In [ ]:
# Load classified projects
housing_path = DATA_DIR / 'housing_projects_classified.csv'
if not housing_path.exists():
    housing_path = Path(CONFIG['paths']['housing_projects'])

df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    display(df[['address_display', 'status', 'net_units']].head(10))

## 3. Calculate Progress Percentage

In [ ]:
# Example progress calculation
test_inspections = [
    ['Foundation'],
    ['Foundation', 'Framing/Rough'],
    ['Foundation', 'Framing', 'Electrical Rough', 'Plumbing Rough'],
    ['Foundation', 'Framing', 'Drywall'],
    INSPECTION_SEQUENCE,  # All complete
]

print("Progress Calculation Examples:")
print("="*60)

for inspections in test_inspections:
    pct = calculate_progress_percent(inspections)
    print(f"  {inspections[:3]}{'...' if len(inspections) > 3 else ''}: {pct}% complete")

## 4. Identify Stalled Projects

Projects with no activity for 180+ days.

In [ ]:
# Check for stalled projects
print(f"Stalled threshold: {STALLED_THRESHOLD} days")

# TODO: This requires last_action_date column from permit data
# When that data is available:
# stalled = identify_stalled_projects(df, STALLED_THRESHOLD, 'last_action_date')

# For now, identify projects in 'In Review' status (may be stalled)
if df is not None and 'status' in df.columns:
    review_status = ['In Review', 'Under Review', 'Incomplete Pending Applicant']
    potentially_stalled = df[df['status'].isin(review_status)]
    
    print(f"\nProjects potentially stalled (in review status): {len(potentially_stalled)}")
    print(f"Total units in review: {potentially_stalled['net_units'].sum():,.0f}")
    
    if len(potentially_stalled) > 0:
        print("\nLargest projects in review:")
        display(potentially_stalled.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']])

## 5. Projects Near Completion

In [ ]:
# Identify projects with approved/permitted status (likely under construction)
if df is not None and 'status' in df.columns:
    approved_keywords = ['Approved', 'Pending Final Action', 'Final']
    
    near_completion = df[df['status'].str.contains('|'.join(approved_keywords), case=False, na=False)]
    
    print(f"Projects approved/near completion: {len(near_completion)}")
    print(f"Total units: {near_completion['net_units'].sum():,.0f}")
    
    if len(near_completion) > 0:
        print("\nLargest approved projects:")
        display(near_completion.nlargest(10, 'net_units')[['address_display', 'net_units', 'status']])

## 6. Export Progress Data

In [ ]:
# Export progress summary
if df is not None:
    # Add progress category
    def get_progress_category(status):
        status = str(status).lower()
        if 'complete' in status or 'occupancy' in status:
            return 'Completed'
        elif 'approved' in status or 'final action' in status:
            return 'Near Completion'
        elif 'review' in status:
            return 'In Progress'
        else:
            return 'Early Stage'
    
    df['progress_category'] = df['status'].apply(get_progress_category)
    
    output_path = DATA_DIR / 'housing_projects_progress.csv'
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook:
- Defined inspection-based progress calculation
- Identified potentially stalled projects
- Tracked projects near completion

**Next:** Run `C1_pipeline_analysis.ipynb` for comprehensive analysis.